In [1]:
%load_ext lab_black

import pandas as pd
import numpy as np
import polars as pl
from tqdm import tqdm
import os
import gc
import warnings

warnings.filterwarnings("ignore")

In [2]:
catalog.exists("sample_submission")

True

In [3]:
sample_submission = catalog.load("sample_submission")
pl_sample_submission = pl.from_pandas(sample_submission)

[12/04/22 13:35:14] INFO     Loading data from 'sample_submission' (CSVDataSet)...              ]8;id=949463;file:///media/mikhail/disk/ML/compet/kaggle/otto-recommender-system/venv_otto/lib/python3.7/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=688762;file:///media/mikhail/disk/ML/compet/kaggle/otto-recommender-system/venv_otto/lib/python3.7/site-packages/kedro/io/data_catalog.py#344\344]8;;\

In [4]:
sample_submission

,session_type,labels
0,12899779_clicks,129004 126836 118524
1,12899779_carts,129004 126836 118524
2,12899779_orders,129004 126836 118524
3,12899780_clicks,129004 126836 118524
4,12899780_carts,129004 126836 118524
...,...,...
5015404,14571580_carts,129004 126836 118524
5015405,14571580_orders,129004 126836 118524
5015406,14571581_clicks,129004 126836 118524
5015407,14571581_carts,129004 126836 118524


In [5]:
pl_sample_submission

session_type,labels
str,str
"""12899779_click...","""129004 126836 ..."
"""12899779_carts...","""129004 126836 ..."
"""12899779_order...","""129004 126836 ..."
"""12899780_click...","""129004 126836 ..."
"""12899780_carts...","""129004 126836 ..."
"""12899780_order...","""129004 126836 ..."
"""12899781_click...","""129004 126836 ..."
"""12899781_carts...","""129004 126836 ..."
"""12899781_order...","""129004 126836 ..."


In [6]:
%%time

TRAIN_PATH = "../data/01_raw/train.jsonl"
train = pl.read_json(TRAIN_PATH, json_lines=True)

TEST_PATH = "../data/01_raw/test.jsonl"
test = pl.read_json(TEST_PATH, json_lines=True)

CPU times: user 7min 17s, sys: 1min 22s, total: 8min 40s
Wall time: 20.4 s


In [7]:
train.head()

session,events
i64,list[struct[3]]
0,"[{1517085,1659304800025,""clicks""}, {1563459,1659304904511,""clicks""}, ... {161938,1661684983707,""clicks""}]"
1,"[{424964,1659304800025,""carts""}, {1492293,1659304852871,""clicks""}, ... {497868,1661714854992,""clicks""}]"
2,"[{763743,1659304800038,""clicks""}, {137492,1659304866606,""clicks""}, ... {672473,1661714215659,""clicks""}]"
3,"[{1425967,1659304800095,""carts""}, {1425967,1659304821062,""clicks""}, ... {925352,1661109666771,""carts""}]"
4,"[{613619,1659304800119,""clicks""}, {298827,1659304836708,""clicks""}, ... {479396,1661586681303,""clicks""}]"


In [8]:
test.head()

session,events
i64,list[struct[3]]
12899779,"[{59625,1661724000278,""clicks""}]"
12899780,"[{1142000,1661724000378,""clicks""}, {582732,1661724058352,""clicks""}, ... {1142000,1661724155248,""clicks""}]"
12899781,"[{141736,1661724000559,""clicks""}, {199008,1661724022851,""clicks""}, ... {918667,1662060160406,""clicks""}]"
12899782,"[{1669402,1661724000568,""clicks""}, {1494780,1661724163530,""clicks""}, ... {1007613,1661803953178,""orders""}]"
12899783,"[{255297,1661724000572,""clicks""}, {1114789,1661724004924,""clicks""}, ... {1817895,1662041140398,""clicks""}]"


In [9]:
train.estimated_size(unit="gb"), test.estimated_size(unit="gb")

(6.231463671661913, 0.21795047726482153)

In [10]:
len(train), len(test)

(12899779, 1671803)

In [11]:
%%time

event_dict = {
    'session': list(),
    'aid': list(),
    'ts': list(),
    'type': list(),
}

events = train.select("events")
sessions = train.select("session")

for i in tqdm(range(len(train))):
    session_current = sessions.row(i)[0]
    events_current = events.row(i)[0]
    for event in events_current:
        event_dict['session'].append(session_current)
        event_dict['aid'].append(event['aid'])
        event_dict['ts'].append(event['ts'])
        event_dict['type'].append(event['type'])

df_train = pl.from_dict(event_dict)

del event_dict
gc.collect()

100%|████████████████████████████| 12899779/12899779 [04:53<00:00, 44008.53it/s]


CPU times: user 5min 55s, sys: 40.2 s, total: 6min 35s
Wall time: 6min 35s


0

In [12]:
%%time

event_dict = {
    'session': list(),
    'aid': list(),
    'ts': list(),
    'type': list(),
}

events = test.select("events")
sessions = test.select("session")

for i in tqdm(range(len(test))):
    session_current = sessions.row(i)[0]
    events_current = events.row(i)[0]
    for event in events_current:
        event_dict['session'].append(session_current)
        event_dict['aid'].append(event['aid'])
        event_dict['ts'].append(event['ts'])
        event_dict['type'].append(event['type'])

df_test = pl.from_dict(event_dict)

del event_dict
gc.collect()

100%|████████████████████████████████| 1671803/1671803 [31:48<00:00, 875.87it/s]


CPU times: user 1min 23s, sys: 30min 30s, total: 31min 53s
Wall time: 31min 50s


0

In [15]:
df_train.shape, df_test.shape

((216716096, 4), (6928123, 4))

In [16]:
df_train.head()

session,aid,ts,type
i64,i64,i64,str
0,1517085,1659304800025,"""clicks"""
0,1563459,1659304904511,"""clicks"""
0,1309446,1659367439426,"""clicks"""
0,16246,1659367719997,"""clicks"""
0,1781822,1659367871344,"""clicks"""


In [17]:
df_test.head()

session,aid,ts,type
i64,i64,i64,str
12899779,59625,1661724000278,"""clicks"""
12899780,1142000,1661724000378,"""clicks"""
12899780,582732,1661724058352,"""clicks"""
12899780,973453,1661724109199,"""clicks"""
12899780,736515,1661724136868,"""clicks"""


In [18]:
%%time

df_train.write_parquet('../data/02_intermediate/df_train.parquet')

CPU times: user 23.8 s, sys: 8.26 s, total: 32 s
Wall time: 25.8 s


In [19]:
%%time

df_test.write_parquet('../data/02_intermediate/df_test.parquet')

CPU times: user 801 ms, sys: 192 ms, total: 993 ms
Wall time: 838 ms
